# AI Hallucination Detection System
**Goal:** Build an explainable baseline that flags answer sentences that may not be supported by a supplied context.

Includes synthetic examples, TF-IDF cosine similarity, threshold selection, evaluation metrics, sentence-level inspection, and JSON/Markdown report export.

> **Limitation:** This is a heuristic starter, not a definitive hallucination detector. Text overlap does not prove factual entailment. Human review is recommended for important decisions.

## 1. Install dependencies
Run in Google Colab or Jupyter.

In [ ]:
%pip install -q pandas scikit-learn matplotlib tabulate

## 2. Imports and configuration

In [ ]:
import json, re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.metrics.pairwise import cosine_similarity

RANDOM_STATE = 42
OUTPUT_DIR = Path("hallucination_detection_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Setup complete.")

## 3. Create an illustrative dataset
These small synthetic examples demonstrate the workflow; they are not a benchmark.

In [ ]:
examples = [
    {"id":"ex01","context":"The Northwind Library opened in 1998. It is located in Chennai and has a collection of 25,000 books.","answer":"The Northwind Library opened in 1998 and is located in Chennai.","label":"grounded"},
    {"id":"ex02","context":"The Bluebird service runs every weekday from 8 AM to 6 PM.","answer":"The Bluebird service runs every day, including Sundays, until midnight.","label":"potential_hallucination"},
    {"id":"ex03","context":"Asha Rao published the report in 2022. The report examines urban tree cover.","answer":"Asha Rao published a report in 2022 about urban tree cover.","label":"grounded"},
    {"id":"ex04","context":"The product weighs 1.4 kilograms and comes with a two-year warranty.","answer":"The product weighs 1.4 kilograms and includes a five-year warranty.","label":"potential_hallucination"},
    {"id":"ex05","context":"The workshop is scheduled for 14 November at the Civic Hall.","answer":"The workshop will take place on 14 November at the Civic Hall.","label":"grounded"},
    {"id":"ex06","context":"The Riverdale team scored 3 goals in the final match.","answer":"The Riverdale team scored 5 goals and won the final match.","label":"potential_hallucination"},
    {"id":"ex07","context":"The plant grows best in indirect sunlight and should be watered when the top layer of soil is dry.","answer":"Keep the plant in indirect sunlight and water it when the soil surface dries.","label":"grounded"},
    {"id":"ex08","context":"The company was founded in 2015 by two engineers.","answer":"The company was founded in 2015 by two engineers and is headquartered in Mumbai.","label":"potential_hallucination"},
    {"id":"ex09","context":"The course contains six modules and is delivered online.","answer":"The online course contains six modules.","label":"grounded"},
    {"id":"ex10","context":"The museum is closed on Mondays. Admission is free for children under five.","answer":"The museum is open every Monday, and children under five pay full admission.","label":"potential_hallucination"},
    {"id":"ex11","context":"The API accepts JSON requests and returns JSON responses.","answer":"The API accepts JSON requests and returns JSON responses.","label":"grounded"},
    {"id":"ex12","context":"The garden contains roses, jasmine, and hibiscus.","answer":"The garden contains roses, jasmine, hibiscus, and tulips.","label":"potential_hallucination"},
]
df = pd.DataFrame(examples)
display(df)

## 4. Split the data
A tiny split demonstrates the workflow only; metrics will be unstable.

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.33, random_state=RANDOM_STATE, stratify=df["label"]
)
print("Train:", len(train_df), "| Test:", len(test_df))
print(train_df["label"].value_counts())

## 5. Sentence-level evidence scoring
TF-IDF cosine similarity estimates lexical overlap between each answer sentence and the context. It does not establish logical entailment.

In [ ]:
def split_sentences(text: str) -> list[str]:
    text = re.sub(r"\\s+", " ", str(text)).strip()
    if not text:
        return []
    return [p.strip() for p in re.split(r"(?<=[.!?])\\s+", text) if p.strip()]

def similarity_scores(context: str, sentences: list[str]) -> list[float]:
    if not sentences:
        return []
    matrix = TfidfVectorizer(
        ngram_range=(1, 2), stop_words="english"
    ).fit_transform([context] + sentences)
    return [float(x) for x in cosine_similarity(matrix[1:], matrix[0:1]).ravel()]

def inspect_answer(context: str, answer: str, threshold: float = 0.16) -> dict[str, Any]:
    sentences = split_sentences(answer)
    scores = similarity_scores(context, sentences)
    details = [
        {"sentence": sentence, "similarity": round(score, 4),
         "potentially_unsupported": bool(score < threshold)}
        for sentence, score in zip(sentences, scores)
    ]
    flagged = [item for item in details if item["potentially_unsupported"]]
    return {
        "threshold": threshold,
        "sentence_count": len(sentences),
        "flagged_sentence_count": len(flagged),
        "potential_hallucination": bool(flagged),
        "sentences": details,
        "method": "TF-IDF cosine similarity heuristic",
    }

inspect_answer(
    "The product weighs 1.4 kilograms and comes with a two-year warranty.",
    "The product weighs 1.4 kilograms and includes a five-year warranty."
)

## 6. Select a threshold on the training set
Threshold tuning on this tiny synthetic set is illustrative only.

In [ ]:
THRESHOLDS = [0.08, 0.12, 0.16, 0.20, 0.24, 0.28]

def predict_label(context: str, answer: str, threshold: float) -> str:
    result = inspect_answer(context, answer, threshold)
    return "potential_hallucination" if result["potential_hallucination"] else "grounded"

rows = []
for threshold in THRESHOLDS:
    preds = [predict_label(r.context, r.answer, threshold)
             for r in train_df.itertuples(index=False)]
    truth = train_df["label"].tolist()
    rows.append({
        "threshold": threshold,
        "precision": precision_score(truth, preds, pos_label="potential_hallucination", zero_division=0),
        "recall": recall_score(truth, preds, pos_label="potential_hallucination", zero_division=0),
        "f1": f1_score(truth, preds, pos_label="potential_hallucination", zero_division=0),
    })
threshold_df = pd.DataFrame(rows)
display(threshold_df)
best_threshold = float(threshold_df.sort_values(["f1", "precision"], ascending=False).iloc[0]["threshold"])
print("Selected training threshold:", best_threshold)

## 7. Evaluate on the holdout set

In [ ]:
test_predictions = [
    predict_label(r.context, r.answer, best_threshold)
    for r in test_df.itertuples(index=False)
]
y_true = test_df["label"].tolist()
metrics = {
    "accuracy": float(accuracy_score(y_true, test_predictions)),
    "precision_hallucination": float(precision_score(y_true, test_predictions, pos_label="potential_hallucination", zero_division=0)),
    "recall_hallucination": float(recall_score(y_true, test_predictions, pos_label="potential_hallucination", zero_division=0)),
    "f1_hallucination": float(f1_score(y_true, test_predictions, pos_label="potential_hallucination", zero_division=0)),
}
print(json.dumps(metrics, indent=2))
print(classification_report(y_true, test_predictions, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_true, test_predictions,
    labels=["grounded", "potential_hallucination"],
    xticks_rotation=20
)
plt.title("Holdout Evaluation — Synthetic Demonstration Data")
plt.tight_layout()
plt.show()

evaluation_df = test_df[["id", "context", "answer", "label"]].copy()
evaluation_df["predicted_label"] = test_predictions
display(evaluation_df)

## 8. Inspect sentence-level flags

In [ ]:
for row in test_df.itertuples(index=False):
    result = inspect_answer(row.context, row.answer, best_threshold)
    print("=" * 80)
    print("ID:", row.id, "| Expected:", row.label)
    print("Predicted:", "potential_hallucination" if result["potential_hallucination"] else "grounded")
    for item in result["sentences"]:
        print(f"score={item['similarity']:.3f} | flagged={item['potentially_unsupported']} | {item['sentence']}")

## 9. Try your own context and answer

In [ ]:
new_context = "The public library opens at 9 AM from Tuesday to Saturday. It is closed on Sundays and Mondays."
new_answer = "The library opens at 9 AM on Wednesdays. The library is open every Sunday."
new_result = inspect_answer(new_context, new_answer, threshold=best_threshold)
print(json.dumps(new_result, indent=2))

## 10. Export JSON and Markdown reports

In [ ]:
report = {
    "project": "AI Hallucination Detection System",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "method": "TF-IDF cosine similarity heuristic",
    "selected_threshold": best_threshold,
    "metrics": metrics,
    "holdout_predictions": evaluation_df.to_dict(orient="records"),
    "limitations": [
        "Synthetic dataset is very small and not representative.",
        "Lexical similarity does not prove factual entailment.",
        "Paraphrases may be incorrectly flagged.",
        "High-overlap contradictions may not be detected.",
        "Human review is recommended for important decisions."
    ],
}
json_path = OUTPUT_DIR / "hallucination_detection_report.json"
md_path = OUTPUT_DIR / "hallucination_detection_report.md"
json_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
markdown_report = (
    "# AI Hallucination Detection Report\n\n"
    f"- Generated (UTC): {report['generated_at_utc']}\n"
    f"- Method: {report['method']}\n"
    f"- Selected threshold: {best_threshold}\n\n"
    "## Holdout metrics\n\n"
    + "\n".join(f"- **{k.replace('_', ' ').title()}**: {v:.3f}" for k, v in metrics.items())
    + "\n\n## Limitations\n\n"
    + "\n".join(f"- {item}" for item in report["limitations"])
)
md_path.write_text(markdown_report, encoding="utf-8")
print("Saved:", json_path.resolve())
print("Saved:", md_path.resolve())

## 11. Next improvements
1. Use sentence embeddings for semantic similarity.
2. Add a natural-language inference (NLI) model to classify entailment, contradiction, and neutrality.
3. Extract atomic claims and verify each claim independently.
4. Add source citations and evidence-span matching.
5. Use a larger, expert-labelled dataset and an untouched test set.
6. Add a Streamlit UI or FastAPI endpoint.
7. Track regressions, latency, and false-positive/false-negative rates.

**Responsible use:** A flagged statement means “review this claim,” not “this claim is definitely false.” Do not use this starter notebook as the sole basis for medical, legal, financial, safety, or other high-impact decisions.